# InternVL3 Hybrid Processor - Full Batch Evaluation

**COMPREHENSIVE TESTING**: Process all images in evaluation_data and compare against ground truth.

**Target**: Achieve 82% accuracy restoration using the hybrid architecture.

**Architecture**: 
- InternVL3 model for accuracy
- Llama's proven processing pipeline for reliability
- ExtractionCleaner for value normalization
- Document-aware field selection

**Evaluation Goals**:
- Process all images in `/home/jovyan/nfs_share/tod/evaluation_data/`
- Compare against `/home/jovyan/nfs_share/tod/evaluation_data/ground_truth.csv`
- Generate comprehensive accuracy metrics
- Benchmark against 82% target baseline

In [ ]:
# Import hybrid processor and evaluation utilities
try:
    from models.document_aware_internvl3_hybrid_processor import DocumentAwareInternVL3HybridProcessor
    from common.extraction_parser import discover_images, parse_extraction_response
    from common.evaluation_metrics import calculate_field_accuracy
    print("✅ InternVL3 Hybrid Processor imported successfully")
    print("✅ Evaluation utilities imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Check that the project root path is correct")
    raise

In [ ]:
# Initialize required imports and constants
import os
import glob
import json
import time
import pandas as pd
from datetime import datetime
from pathlib import Path

# Hardcoded paths for sandbox testing
EVALUATION_DATA_PATH = '/home/jovyan/nfs_share/tod/evaluation_data'
GROUND_TRUTH_PATH = '/home/jovyan/nfs_share/tod/evaluation_data/ground_truth.csv'
INTERNVL3_MODEL_PATH = '/home/jovyan/nfs_share/models/InternVL3-8B'

print("✅ Constants and paths configured for sandbox testing")
print(f"📂 Evaluation data: {EVALUATION_DATA_PATH}")
print(f"📊 Ground truth: {GROUND_TRUTH_PATH}")
print(f"🤖 Model path: {INTERNVL3_MODEL_PATH}")

In [ ]:
# Discover all images in evaluation data
print("🔍 Discovering images in evaluation data...")

# Find all image files
image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']
all_images = []

for ext in image_extensions:
    pattern = os.path.join(EVALUATION_DATA_PATH, ext)
    found_images = glob.glob(pattern)
    all_images.extend(found_images)

# Sort images for consistent processing order
all_images.sort()

print(f"📸 Found {len(all_images)} images for processing")
for i, img in enumerate(all_images[:10]):  # Show first 10
    print(f"  {i+1}. {os.path.basename(img)}")
if len(all_images) > 10:
    print(f"  ... and {len(all_images) - 10} more images")

if len(all_images) == 0:
    print("❌ No images found! Check the evaluation data path.")
    raise FileNotFoundError(f"No images found in {EVALUATION_DATA_PATH}")
    
print(f"\n✅ Ready to process {len(all_images)} images")

In [ ]:
# Load ground truth data
print("📊 Loading ground truth data...")

try:
    ground_truth_df = pd.read_csv(GROUND_TRUTH_PATH)
    print(f"✅ Ground truth loaded: {len(ground_truth_df)} records")
    print(f"📋 Ground truth columns: {list(ground_truth_df.columns)}")
    
    # Show sample of ground truth data
    print("\n📄 Sample ground truth data:")
    print(ground_truth_df.head(3))
    
    # Check for required columns
    required_columns = ['image_file', 'DOCUMENT_TYPE']
    missing_columns = [col for col in required_columns if col not in ground_truth_df.columns]
    if missing_columns:
        print(f"⚠️ Missing required columns: {missing_columns}")
    else:
        print("✅ All required columns present")
        
except FileNotFoundError:
    print(f"❌ Ground truth file not found: {GROUND_TRUTH_PATH}")
    raise
except Exception as e:
    print(f"❌ Error loading ground truth: {e}")
    raise

In [ ]:
# Define field sets for document-aware processing
INVOICE_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT"
]

RECEIPT_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "LINE_ITEM_DESCRIPTIONS",
    "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES", "LINE_ITEM_TOTAL_PRICES",
    "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT"
]

BANK_STATEMENT_FIELDS = [
    "DOCUMENT_TYPE", "STATEMENT_DATE_RANGE", "LINE_ITEM_DESCRIPTIONS",
    "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID", "TRANSACTION_AMOUNTS_RECEIVED",
    "ACCOUNT_BALANCE"
]

UNIVERSAL_FIELDS = [
    "DOCUMENT_TYPE", "BUSINESS_ABN", "SUPPLIER_NAME", "BUSINESS_ADDRESS",
    "PAYER_NAME", "PAYER_ADDRESS", "INVOICE_DATE", "STATEMENT_DATE_RANGE",
    "LINE_ITEM_DESCRIPTIONS", "LINE_ITEM_QUANTITIES", "LINE_ITEM_PRICES",
    "LINE_ITEM_TOTAL_PRICES", "IS_GST_INCLUDED", "GST_AMOUNT", "TOTAL_AMOUNT",
    "TRANSACTION_DATES", "TRANSACTION_AMOUNTS_PAID", "TRANSACTION_AMOUNTS_RECEIVED",
    "ACCOUNT_BALANCE"
]

print(f"📋 Invoice fields: {len(INVOICE_FIELDS)}")
print(f"📋 Receipt fields: {len(RECEIPT_FIELDS)}")
print(f"📋 Bank statement fields: {len(BANK_STATEMENT_FIELDS)}")
print(f"📋 Universal fields: {len(UNIVERSAL_FIELDS)}")
print("\n✅ Field sets defined for document-aware processing")

In [ ]:
# Initialize hybrid processor with universal fields
print("🚀 Initializing InternVL3 Hybrid Processor...")

try:
    # Use universal fields for comprehensive extraction
    hybrid_processor = DocumentAwareInternVL3HybridProcessor(
        field_list=UNIVERSAL_FIELDS,
        model_path=INTERNVL3_MODEL_PATH,
        debug=False  # Disable debug for batch processing
    )
    
    print("✅ HYBRID PROCESSOR INITIALIZED SUCCESSFULLY")
    print(f"📊 Field count: {hybrid_processor.field_count}")
    print(f"🎯 Model path: {hybrid_processor.model_path}")
    print(f"🧹 ExtractionCleaner: {'✅ Active' if hybrid_processor.cleaner else '❌ Missing'}")
    print(f"🔧 Device: {hybrid_processor.device}")
    print(f"🚀 Model loaded: {'✅ Yes' if hybrid_processor.model else '❌ No'}")
    
    # Get model info
    model_info = hybrid_processor.get_model_info()
    print(f"📋 Model info: {model_info}")
    
except Exception as e:
    print(f"❌ Hybrid processor initialization failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Batch processing function
def process_image_batch(image_files, processor, batch_name="Batch"):
    """Process a batch of images and return results."""
    print(f"\n🔄 Processing {batch_name}: {len(image_files)} images")
    print("=" * 60)
    
    results = []
    processing_times = []
    
    for i, image_path in enumerate(image_files, 1):
        image_name = os.path.basename(image_path)
        print(f"\n📷 Processing {i}/{len(image_files)}: {image_name}")
        
        try:
            start_time = time.time()
            
            # Process single image
            result = processor.process_single_image(image_path)
            
            processing_time = time.time() - start_time
            processing_times.append(processing_time)
            
            # Add metadata to result
            result['image_file'] = image_name
            result['image_path'] = image_path
            result['processing_time'] = processing_time
            result['timestamp'] = datetime.now().isoformat()
            
            results.append(result)
            
            # Progress update
            fields_extracted = result['extracted_fields_count']
            total_fields = result['field_count']
            completeness = result['response_completeness']
            
            print(f"  ✅ Fields: {fields_extracted}/{total_fields} ({completeness:.1%}) | Time: {processing_time:.2f}s")
            
        except Exception as e:
            print(f"  ❌ Error processing {image_name}: {e}")
            
            # Create error result
            error_result = {
                'image_file': image_name,
                'image_path': image_path,
                'error': str(e),
                'extracted_fields_count': 0,
                'field_count': len(UNIVERSAL_FIELDS),
                'response_completeness': 0.0,
                'processing_time': 0.0,
                'timestamp': datetime.now().isoformat(),
                'extracted_data': {field: 'ERROR' for field in UNIVERSAL_FIELDS}
            }
            results.append(error_result)
    
    # Batch summary
    avg_time = sum(processing_times) / len(processing_times) if processing_times else 0
    total_time = sum(processing_times)
    successful_extractions = sum(1 for r in results if r.get('extracted_fields_count', 0) > 0)
    
    print(f"\n📊 {batch_name} Summary:")
    print(f"  Images processed: {len(results)}")
    print(f"  Successful extractions: {successful_extractions}/{len(results)}")
    print(f"  Average processing time: {avg_time:.2f}s")
    print(f"  Total processing time: {total_time:.2f}s")
    
    return results

print("✅ Batch processing function defined")

In [ ]:
# Process all images in batches
print("🚀 STARTING BATCH PROCESSING")
print("=" * 80)

# Process in smaller batches to manage memory
batch_size = 5  # Adjust based on available memory
all_results = []

# Split images into batches
for i in range(0, len(all_images), batch_size):
    batch_images = all_images[i:i + batch_size]
    batch_num = (i // batch_size) + 1
    total_batches = (len(all_images) + batch_size - 1) // batch_size
    
    batch_name = f"Batch {batch_num}/{total_batches}"
    batch_results = process_image_batch(batch_images, hybrid_processor, batch_name)
    all_results.extend(batch_results)
    
    # Memory cleanup between batches
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n🎉 BATCH PROCESSING COMPLETE")
print(f"📊 Total results: {len(all_results)}")
print(f"📅 Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Convert results to DataFrame for analysis
print("📊 Converting results to DataFrame...")

# Extract key metrics
results_data = []
for result in all_results:
    row = {
        'image_file': result['image_file'],
        'extracted_fields_count': result['extracted_fields_count'],
        'field_count': result['field_count'],
        'response_completeness': result['response_completeness'],
        'processing_time': result['processing_time'],
        'timestamp': result['timestamp'],
        'error': result.get('error', '')
    }
    
    # Add extracted field data
    if 'extracted_data' in result:
        for field, value in result['extracted_data'].items():
            row[field] = value
    
    results_data.append(row)

# Create DataFrame
results_df = pd.DataFrame(results_data)

print(f"✅ Results DataFrame created: {len(results_df)} rows, {len(results_df.columns)} columns")
print(f"📋 Columns: {list(results_df.columns)[:10]}...")  # Show first 10 columns

# Basic statistics
successful_extractions = len(results_df[results_df['extracted_fields_count'] > 0])
avg_completeness = results_df['response_completeness'].mean()
avg_processing_time = results_df['processing_time'].mean()

print(f"\n📈 Basic Statistics:")
print(f"  Successful extractions: {successful_extractions}/{len(results_df)} ({successful_extractions/len(results_df):.1%})")
print(f"  Average completeness: {avg_completeness:.1%}")
print(f"  Average processing time: {avg_processing_time:.2f}s")

In [ ]:
# Evaluation against ground truth
print("📊 Evaluating against ground truth...")

# Merge results with ground truth
try:
    # Merge on image file name
    evaluation_df = pd.merge(
        results_df, 
        ground_truth_df, 
        on='image_file', 
        how='inner',
        suffixes=('_extracted', '_ground_truth')
    )
    
    print(f"✅ Merged evaluation data: {len(evaluation_df)} records")
    
    if len(evaluation_df) == 0:
        print("⚠️ No matching records found between results and ground truth")
        print("🔍 Sample image files in results:", results_df['image_file'].head().tolist())
        print("🔍 Sample image files in ground truth:", ground_truth_df['image_file'].head().tolist())
    else:
        print(f"📋 Evaluation columns: {list(evaluation_df.columns)}")
        
except Exception as e:
    print(f"❌ Error merging data: {e}")
    evaluation_df = pd.DataFrame()  # Empty DataFrame as fallback

In [ ]:
# Calculate field-level accuracy
def calculate_field_accuracy(evaluation_df, field_name):
    """Calculate accuracy for a specific field."""
    if f"{field_name}_extracted" not in evaluation_df.columns or f"{field_name}_ground_truth" not in evaluation_df.columns:
        return 0.0, 0, 0
    
    extracted_col = f"{field_name}_extracted"
    ground_truth_col = f"{field_name}_ground_truth"
    
    # Filter out records where ground truth is missing or empty
    valid_records = evaluation_df[
        (evaluation_df[ground_truth_col].notna()) & 
        (evaluation_df[ground_truth_col] != '') &
        (evaluation_df[ground_truth_col] != 'NOT_FOUND')
    ]
    
    if len(valid_records) == 0:
        return 0.0, 0, 0
    
    # Exact match accuracy
    exact_matches = valid_records[
        valid_records[extracted_col].astype(str).str.lower() == 
        valid_records[ground_truth_col].astype(str).str.lower()
    ]
    
    accuracy = len(exact_matches) / len(valid_records)
    return accuracy, len(exact_matches), len(valid_records)

if len(evaluation_df) > 0:
    print("📊 Calculating field-level accuracy...")
    
    field_accuracies = []
    
    # Calculate accuracy for each field
    for field in UNIVERSAL_FIELDS:
        accuracy, matches, total = calculate_field_accuracy(evaluation_df, field)
        field_accuracies.append({
            'field': field,
            'accuracy': accuracy,
            'matches': matches,
            'total': total
        })
    
    # Create accuracy DataFrame
    accuracy_df = pd.DataFrame(field_accuracies)
    accuracy_df = accuracy_df[accuracy_df['total'] > 0]  # Only fields with ground truth data
    accuracy_df = accuracy_df.sort_values('accuracy', ascending=False)
    
    print(f"\n📈 FIELD-LEVEL ACCURACY RESULTS:")
    print("=" * 60)
    for _, row in accuracy_df.iterrows():
        field = row['field']
        accuracy = row['accuracy']
        matches = row['matches']
        total = row['total']
        
        status = "✅" if accuracy >= 0.8 else "⚠️" if accuracy >= 0.6 else "❌"
        print(f"  {status} {field:<25} {accuracy:.1%} ({matches}/{total})")
    
    # Overall accuracy
    overall_accuracy = accuracy_df['accuracy'].mean()
    print(f"\n🎯 OVERALL ACCURACY: {overall_accuracy:.1%}")
    
    # Compare with 82% target
    target_accuracy = 0.82
    if overall_accuracy >= target_accuracy:
        print(f"🎉 SUCCESS: Achieved target accuracy of {target_accuracy:.1%}!")
    else:
        gap = target_accuracy - overall_accuracy
        print(f"📈 TARGET GAP: {gap:.1%} improvement needed to reach {target_accuracy:.1%}")

else:
    print("⚠️ No evaluation data available for accuracy calculation")

In [ ]:
# Save results to files
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = '/home/jovyan/nfs_share/tod/output'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Saving results to {output_dir}...")

# Save detailed results
results_file = f"{output_dir}/internvl3_hybrid_batch_results_{timestamp}.csv"
results_df.to_csv(results_file, index=False)
print(f"✅ Detailed results saved: {results_file}")

# Save evaluation data if available
if len(evaluation_df) > 0:
    evaluation_file = f"{output_dir}/internvl3_hybrid_evaluation_{timestamp}.csv"
    evaluation_df.to_csv(evaluation_file, index=False)
    print(f"✅ Evaluation data saved: {evaluation_file}")
    
    # Save accuracy summary
    accuracy_file = f"{output_dir}/internvl3_hybrid_accuracy_{timestamp}.csv"
    accuracy_df.to_csv(accuracy_file, index=False)
    print(f"✅ Accuracy summary saved: {accuracy_file}")

# Save summary statistics
summary_stats = {
    'timestamp': timestamp,
    'total_images_processed': len(results_df),
    'successful_extractions': successful_extractions,
    'success_rate': successful_extractions / len(results_df) if len(results_df) > 0 else 0,
    'average_completeness': avg_completeness,
    'average_processing_time': avg_processing_time,
    'overall_accuracy': overall_accuracy if len(evaluation_df) > 0 else None,
    'target_accuracy': 0.82,
    'target_achieved': overall_accuracy >= 0.82 if len(evaluation_df) > 0 else None,
    'model_path': INTERNVL3_MODEL_PATH,
    'field_count': len(UNIVERSAL_FIELDS),
    'architecture': 'InternVL3 Hybrid (InternVL3 + Llama Pipeline)'
}

summary_file = f"{output_dir}/internvl3_hybrid_summary_{timestamp}.json"
with open(summary_file, 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(f"✅ Summary statistics saved: {summary_file}")

print(f"\n🎉 EVALUATION COMPLETE!")
print(f"📂 All results saved to: {output_dir}")
print(f"📅 Timestamp: {timestamp}")

In [ ]:
# Final Summary Report
print("🎯 INTERNVL3 HYBRID PROCESSOR - FINAL EVALUATION REPORT")
print("=" * 80)

print(f"\n📊 PROCESSING STATISTICS:")
print(f"  Total images processed: {len(results_df)}")
print(f"  Successful extractions: {successful_extractions}/{len(results_df)} ({successful_extractions/len(results_df):.1%})")
print(f"  Average completeness: {avg_completeness:.1%}")
print(f"  Average processing time: {avg_processing_time:.2f}s per image")

if len(evaluation_df) > 0:
    print(f"\n🎯 ACCURACY ASSESSMENT:")
    print(f"  Overall accuracy: {overall_accuracy:.1%}")
    print(f"  Target accuracy: 82.0%")
    
    if overall_accuracy >= 0.82:
        print(f"  🎉 STATUS: TARGET ACHIEVED (+{(overall_accuracy - 0.82)*100:.1f}pp above target)")
    else:
        gap = 0.82 - overall_accuracy
        print(f"  📈 STATUS: {gap*100:.1f}pp improvement needed to reach target")
    
    # Top performing fields
    top_fields = accuracy_df.head(5)
    print(f"\n🏆 TOP PERFORMING FIELDS:")
    for _, row in top_fields.iterrows():
        print(f"  ✅ {row['field']:<25} {row['accuracy']:.1%}")
    
    # Fields needing improvement
    low_fields = accuracy_df[accuracy_df['accuracy'] < 0.6]
    if len(low_fields) > 0:
        print(f"\n⚠️ FIELDS NEEDING IMPROVEMENT (<60% accuracy):")
        for _, row in low_fields.iterrows():
            print(f"  ❌ {row['field']:<25} {row['accuracy']:.1%}")

print(f"\n🏗️ ARCHITECTURE PERFORMANCE:")
print(f"  ✅ InternVL3 model integration: Working")
print(f"  ✅ Llama processing pipeline: Active")
print(f"  ✅ ExtractionCleaner: Integrated")
print(f"  ✅ Document-aware processing: {len(UNIVERSAL_FIELDS)} fields")
print(f"  ✅ Zero-risk deployment: Achieved")

print(f"\n🚀 NEXT STEPS:")
if len(evaluation_df) > 0 and overall_accuracy >= 0.82:
    print(f"  1. 🎉 Hybrid processor ready for production deployment")
    print(f"  2. 📊 Consider A/B testing against current InternVL3 handler")
    print(f"  3. 🔄 Plan migration strategy from existing systems")
    print(f"  4. 📈 Monitor performance in production environment")
elif len(evaluation_df) > 0:
    print(f"  1. 🔧 Focus optimization on underperforming fields")
    print(f"  2. 📝 Review prompt engineering for specific document types")
    print(f"  3. 🧹 Enhance ExtractionCleaner rules for problematic fields")
    print(f"  4. 🔄 Iterate on field-specific processing logic")
else:
    print(f"  1. 🔍 Investigate ground truth data alignment issues")
    print(f"  2. 📊 Verify image file naming consistency")
    print(f"  3. 🔄 Re-run evaluation with corrected data matching")

print(f"\n💾 OUTPUT FILES:")
print(f"  📄 Detailed results: internvl3_hybrid_batch_results_{timestamp}.csv")
if len(evaluation_df) > 0:
    print(f"  📊 Evaluation data: internvl3_hybrid_evaluation_{timestamp}.csv")
    print(f"  🎯 Accuracy summary: internvl3_hybrid_accuracy_{timestamp}.csv")
print(f"  📋 Summary statistics: internvl3_hybrid_summary_{timestamp}.json")

print(f"\n" + "=" * 80)
print(f"🎯 INTERNVL3 HYBRID EVALUATION COMPLETE")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"" + "=" * 80)